# 16. Cost: training time, prediction time and size
**Question:** what does each method cost to train and to run? (for the report's discussion of complexity)
**Measured:** training seconds, prediction seconds for all test rows (264,000), milliseconds for a single row, and size in memory (pickled) for Models A, B, C; the simple rules; and the per-building Isolation Forest (raw variant, 100 trees x 199 buildings).
**Writes:** `results/complexity/complexity_measurements.csv`. Timings depend on the machine; the reported numbers were taken on a shared 2-core cloud machine (A/B/C train about 39.6/40.5/43.4 s, score about 3 s for 264,000 rows, about 10 MB each; typical value 0.16 MB; Isolation Forest 19.5 s to train, 2.1 s to score, about 219 MB for 199x100 trees). Expect other numbers on your computer; the ordering is what matters.

In [ ]:
from config import *
import numpy as np, pandas as pd
RESULTS = ROOT / "results" / "complexity"
RESULTS.mkdir(parents=True, exist_ok=True)
O = str(OUT_DIR) + "/"          # results of notebooks 01-10 (run those first)
def read_any(name):
    """Read name.csv.gz or name.csv or name.parquet from OUT_DIR."""
    for ext in (".csv.gz", ".csv", ".parquet"):
        p = OUT_DIR / (name + ext)
        if p.exists():
            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
    raise FileNotFoundError(name)
import time, pickle

In [ ]:
# This notebook re-uses the feature building and model fitting of notebook 03 (cells up to the point where predictions are saved).
# Its own output folder is redirected so nothing of notebook 03's is overwritten.
import json, config
import shutil
config.OUT_DIR = RESULTS / "_scratch"; config.OUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(OUT_DIR / "labeled_building_ids.csv", config.OUT_DIR)   # the only file of notebook 01 that notebook 03 reads
nb03 = json.load(open("03_gepiii_predictors.ipynb"))
ns = {"__name__": "__main__"}
def run_nb03(stop_when):
    for c in nb03["cells"]:
        if c["cell_type"] != "code": continue
        s = "".join(c["source"])
        if stop_when(s): break
        exec(compile(s, "nb03", "exec"), ns)

In [ ]:
run_nb03(lambda s: 'to_parquet' in s and 'predictor_settings' in s)

In [ ]:
df=ns['df'];FEAT=ns['FEATURES'];BEST=ns['BEST'];fit=ns['fit'];train_rows=ns['train_rows']
test=df[(df.split=='test')]
rows=[]
import os as _o
print('CPU count',_o.cpu_count(),flush=True)
for k in 'ABC':
    f=FEAT[k]
    t0=time.perf_counter();m=fit(f,BEST,train_rows);tr=time.perf_counter()-t0
    X=test[f]
    t0=time.perf_counter();p=m.predict(X);pr=time.perf_counter()-t0
    X1=X.iloc[:1000]
    t0=time.perf_counter()
    for i in range(200): m.predict(X1.iloc[i:i+1])
    one=(time.perf_counter()-t0)/200
    rows.append(dict(method=f'Model {k}',features=len(f),train_rows=len(train_rows),train_seconds=round(tr,1),predict_rows=len(X),predict_seconds_all=round(pr,2),predict_microsec_per_row_batch=round(pr/len(X)*1e6,1),predict_ms_single_row=round(one*1e3,2),model_size_MB=round(len(pickle.dumps(m))/1e6,2),n_trees=int(m.n_iter_)))
    print(rows[-1],flush=True)

model_rows=rows

In [ ]:
# simple rules and Isolation Forest (raw variant), from saved predictions and labels
from sklearn.ensemble import IsolationForest
pr=read_any('predictions');pr['timestamp']=pd.to_datetime(pr.timestamp);lab=read_any('lead_labels')[['building_id','timestamp','in_zero_run']];lab['timestamp']=pd.to_datetime(lab.timestamp)
d=pr.merge(lab,on=['building_id','timestamp'],how='left').sort_values(['building_id','timestamp']).reset_index(drop=True)
d['in_zero_run']=d.in_zero_run.fillna(False).astype(bool);d['y']=d.meter_reading.where(~d.in_zero_run)
d['hour']=d.timestamp.dt.hour;d['wk']=(d.timestamp.dt.dayofweek>=5).astype(int)
t0=time.perf_counter();_=d.groupby('building_id')['y'].shift(1);last_t=time.perf_counter()-t0
tr=d[(d.split=='train')&d.y.notna()]
t0=time.perf_counter();med3=tr.groupby(['building_id','hour','wk'])['y'].median();typ_t=time.perf_counter()-t0
size_typ=len(pickle.dumps(med3))
d['ly']=np.log1p(d.y);g=d.groupby('building_id')['ly'];d['d1']=d.ly-g.shift(1);d['d24']=d.ly-g.shift(24)
F=['ly','d1','d24','hour','wk'];fit_t=sc_t=size=0;n=0;nb=0
for b,s in d.groupby('building_id'):
    ok=s[F].notna().all(axis=1);trn=s[ok&(s.timestamp<'2016-09-13')]
    if len(trn)<200: continue
    t0=time.perf_counter();m=IsolationForest(n_estimators=100,max_samples=256,random_state=0).fit(trn[F]);fit_t+=time.perf_counter()-t0
    te=s[ok&(s.split=='test')]
    if len(te)==0: continue
    t0=time.perf_counter();m.score_samples(te[F]);sc_t+=time.perf_counter()-t0
    size+=len(pickle.dumps(m));n+=len(te);nb+=1

In [ ]:
rows = model_rows + [
 dict(method='Same as last hour',features=0,train_seconds=0.0,predict_seconds_all=round(last_t,3),predict_rows=len(d),model_size_MB=0.0,n_trees=0),
 dict(method='Historical typical value',features=3,train_seconds=round(typ_t,2),model_size_MB=round(size_typ/1e6,3),n_trees=0),
 dict(method=f'Isolation Forest raw ({nb} buildings x 100 trees)',features=5,train_seconds=round(fit_t,1),predict_seconds_all=round(sc_t,2),predict_rows=n,model_size_MB=round(size/1e6,2),n_trees=100*nb)]
R=pd.DataFrame(rows); R.to_csv(RESULTS/'complexity_measurements.csv',index=False); print(R.to_string())